# GHSL weeks 1–5: completed three-product pilot

Population, built-up surface (total/NRES), and matching-year settlement classes.
The pipeline ran on local HDFS with Spark. These read-only notebook code cells were
executed by the package builder in Python; this is not a Colab or Jupyter-kernel run.
All ten historical epochs plus 2025/2030 projection tests are included.

The area is the Bengaluru test rectangle, not the municipal boundary. National
processing and settlement-change analysis belong to weeks 6–8.

In [1]:
from pathlib import Path
import json
here = Path.cwd().resolve()
ROOT = next(p / 'integration' for p in [here, *here.parents] if (p / 'integration/config/pilot.json').exists())
cfg = json.loads((ROOT / 'config/pilot.json').read_text())
print(cfg['name'], cfg['bbox_wgs84'], cfg['epochs'])

bengaluru_three_products_weeks1_5 [77.3, 12.7, 77.9, 13.3] [1975, 1980, 1985, 1990, 1995, 2000, 2005, 2010, 2015, 2020, 2025, 2030]


## 1. Executed validation evidence

In [2]:
report = json.loads((ROOT / 'reports/spark_hdfs_validation.json').read_text())
print(report['status'], report['spark_version'], report['master'])
print('Analytical rows:', report['analytical_rows'])
print('Coarse rows:', report['coarse_rows'])
print('Rejected rows:', report['rejected_rows'])
print('Incomplete trajectories:', report['incomplete_trajectories'])
print('Checks:', ', '.join(report['checks']))

PASS 3.5.6 local[2]
Analytical rows: 5212356
Coarse rows: 53460
Rejected rows: 0
Incomplete trajectories: 0
Checks: source schemas/values/epochs, unique keys, bidirectional coverage joins, native counts and inherited matching-year classes, three-product conservation, coarse coverage/conservation, full-content Parquet round trips


## 2. Read the partitioned three-product handoff

In [3]:
import pyarrow.dataset as ds
path = ROOT / 'data/curated/grid_year'
assert (path / '_INTEGRATION_SUCCESS').exists()
dataset = ds.dataset(path / 'analytical_100m', format='parquet', partitioning='hive')
print(dataset.schema)
sample = dataset.scanner(filter=ds.field('year') == 2020,
    columns=['cell_id', 'parent_cell_id', 'year', 'population', 'built_surface_m2',
             'built_surface_nres_m2', 'settlement_class']).head(5)
for row in sample.to_pylist(): print(row)

parent_row: int32
parent_col: int32
source_row: int32
source_col: int32
population: double
built_surface_m2: double
built_surface_nres_m2: double
settlement_class: int32
grid_id: string not null
cell_id: string not null
parent_grid_id: string not null
parent_cell_id: string not null
is_valid: bool not null
settlement_resolution_m: int32 not null
settlement_inherited: bool not null
source_release: string not null
population_version: string not null
built_version: string not null
settlement_version: string not null
epoch_type: string
year: int32
-- schema metadata --
org.apache.spark.version: '3.5.6'
org.apache.spark.sql.parquet.row.metadata: '{"type":"struct","fields":[{"' + 1379
{'cell_id': '73612_256607', 'parent_cell_id': '7361_25660', 'year': 2020, 'population': 0.0, 'built_surface_m2': 0.0, 'built_surface_nres_m2': 0.0, 'settlement_class': 11}
{'cell_id': '73612_256644', 'parent_cell_id': '7361_25664', 'year': 2020, 'population': 0.0712943747639656, 'built_surface_m2': 3.0, 'built_

## 3. Saved Spark SQL summary for 2020

In [4]:
summary = ds.dataset(path / 'sql_class_summary', format='parquet').to_table()
for row in summary.to_pylist():
    if row['year'] == 2020: print(row)

{'year': 2020, 'epoch_type': 'historical', 'settlement_class': 10, 'fine_cells': 200, 'population': 0.0, 'built_surface_m2': 0.0, 'built_surface_nres_m2': 0.0}
{'year': 2020, 'epoch_type': 'historical', 'settlement_class': 11, 'fine_cells': 68925, 'population': 12878.877812696446, 'built_surface_m2': 8267055.0, 'built_surface_nres_m2': 280520.0}
{'year': 2020, 'epoch_type': 'historical', 'settlement_class': 12, 'fine_cells': 125765, 'population': 203994.97645338258, 'built_surface_m2': 33213275.0, 'built_surface_nres_m2': 1358925.0}
{'year': 2020, 'epoch_type': 'historical', 'settlement_class': 13, 'fine_cells': 5745, 'population': 27933.723004904576, 'built_surface_m2': 2432548.0, 'built_surface_nres_m2': 176562.0}
{'year': 2020, 'epoch_type': 'historical', 'settlement_class': 21, 'fine_cells': 122221, 'population': 826904.3058665863, 'built_surface_m2': 56404017.0, 'built_surface_nres_m2': 2640498.0}
{'year': 2020, 'epoch_type': 'historical', 'settlement_class': 22, 'fine_cells': 324

## 4. Reproduce a fresh Spark run

Run from the project root with the documented `.venv`. Never overwrite delivered data.

```sh
.venv/bin/python integration/src/spark_integrate.py \
  --input integration/data/prepared/tables \
  --output integration/data/curated/notebook_rerun_1 \
  --report integration/reports/notebook_rerun_1.json
```

Prepared inputs and raw raster archives are retained in the development workspace.
They are excluded from the compact handoff ZIP; reproduce them with the download and
preparation scripts before rerunning from the ZIP. Read `integration/README.md` for
HDFS commands, source citations, NoData semantics and scope limitations.